# 🌫️ Air Quality Prediction Using Machine Learning

**Organization:** INLIGHN TECH  
**Project:** Air Quality Prediction | Time Series Forecasting  
**Tools:** Python, pandas, scikit-learn, Prophet, ARIMA, SARIMA, LSTM  

---

## 📌 Project Overview

This notebook builds a comprehensive air quality prediction pipeline using historical pollution data from Indian cities. We:
- Preprocess and explore the dataset
- Engineer time-based and lag features
- Train and compare **8+ models** (Linear Regression, Random Forest, Gradient Boosting, ARIMA, SARIMA, Holt-Winters, Prophet, LSTM)
- Apply **Hyperparameter Tuning** (GridSearchCV / RandomizedSearchCV)
- Evaluate all models with MAE, RMSE, and R²
- Predict air quality using **dummy/synthetic data**
- Discuss **Limitations** and **Conclusions**

---
## 1. Setup & Library Imports

In [ ]:
# ── Install dependencies (uncomment in Colab) ──
# !pip install prophet statsmodels tensorflow scikit-learn pandas numpy matplotlib seaborn flask joblib

import warnings
warnings.filterwarnings('ignore')

# Data manipulation
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# Preprocessing & metrics
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

# Machine Learning models
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression

# Time-series models
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Prophet
try:
    from prophet import Prophet
    PROPHET_AVAILABLE = True
except ImportError:
    print("Prophet not installed. Run: pip install prophet")
    PROPHET_AVAILABLE = False

# Deep learning
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping
    KERAS_AVAILABLE = True
except ImportError:
    print("TensorFlow not installed. Run: pip install tensorflow")
    KERAS_AVAILABLE = False

# Persistence
import joblib, os, json

print("✅ All libraries loaded successfully!")
print(f"  Prophet available : {PROPHET_AVAILABLE}")
print(f"  Keras/TF available: {KERAS_AVAILABLE}")

---
## 2. Load & Explore the Dataset

We load the air pollution CSV containing daily readings across Indian cities. Each row contains pollutant concentrations (CO, NO, NO₂, O₃, SO₂, PM2.5, PM10, NH₃) and an AQI (Air Quality Index) category label.

In [ ]:
# ── Load data ──
# If running in Colab, upload the CSV first then set the path
DATA_PATH = "air_pollution_data.csv"   # adjust path if needed

df_raw = pd.read_csv(DATA_PATH)
print(f"Dataset shape : {df_raw.shape}")
print(f"Columns       : {list(df_raw.columns)}")
df_raw.head()

In [ ]:
# Basic info: data types and non-null counts help us spot columns needing attention
print("=== Data Types & Non-null Counts ===")
df_raw.info()
print("\n=== Statistical Summary ===")
df_raw.describe()

In [ ]:
# Check for missing values and unique cities in the dataset
print("=== Missing Values per Column ===")
print(df_raw.isnull().sum())
print(f"\nUnique cities  : {df_raw['city'].nunique()}")
print(f"Cities         : {sorted(df_raw['city'].unique())}")

---
## 3. Data Preprocessing

Raw sensor data often contains sentinel values (-200) that represent missing or faulty readings. We replace these with `NaN` and impute using the column mean. We also parse the date column properly so we can perform time-series operations.

- **Sentinel replacement**: -200 → NaN → column mean imputation
- **Date parsing**: convert string dates to pandas datetime
- **AQI labelling**: map numeric AQI (1–6) to descriptive categories

In [ ]:
df = df_raw.copy()

# 3.1 Parse date – dayfirst=True handles DD-MM-YYYY format common in Indian datasets
df['date'] = pd.to_datetime(df['date'], dayfirst=True, errors='coerce')
df = df.dropna(subset=['date'])
df = df.sort_values(['city', 'date']).reset_index(drop=True)

# 3.2 Replace sentinel value -200 with NaN, then fill with column mean
pollutant_cols = ['co', 'no', 'no2', 'o3', 'so2', 'pm2_5', 'pm10', 'nh3', 'aqi']
df[pollutant_cols] = df[pollutant_cols].replace(-200, np.nan)
df[pollutant_cols] = df[pollutant_cols].fillna(df[pollutant_cols].mean())

print("After preprocessing:")
print(df.dtypes)
print(df.isnull().sum())
df.head()

In [ ]:
# Map numeric AQI to descriptive labels for better interpretability in plots
aqi_labels = {1: 'Good', 2: 'Satisfactory', 3: 'Moderate', 4: 'Poor', 5: 'Very Poor', 6: 'Severe'}
df['aqi_label'] = df['aqi'].map(aqi_labels)
print(df['aqi_label'].value_counts())

---
## 4. Exploratory Data Analysis (EDA)

Before modelling, we visualize key patterns:
- **AQI distribution**: overall air quality spread across India
- **City-level PM2.5**: which cities are the most polluted
- **Correlation heatmap**: relationships between pollutants
- **Delhi time series**: temporal trend in PM2.5 for our target city

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Air Quality EDA', fontsize=16, fontweight='bold')

# 4a. AQI distribution across the dataset
ax = axes[0, 0]
df['aqi'].value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('AQI Category Distribution')
ax.set_xlabel('AQI')
ax.set_ylabel('Count')
ax.set_xticklabels([aqi_labels.get(i, i) for i in sorted(df['aqi'].unique())], rotation=30)

# 4b. Top 15 cities by average PM2.5 – highlights the most polluted regions
ax = axes[0, 1]
city_pm25 = df.groupby('city')['pm2_5'].mean().sort_values(ascending=False).head(15)
city_pm25.plot(kind='barh', ax=ax, color='tomato')
ax.set_title('Top 15 Cities – Avg PM2.5')
ax.set_xlabel('PM2.5 (µg/m³)')

# 4c. Correlation heatmap – helps select features and spot multicollinearity
ax = axes[1, 0]
corr = df[pollutant_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax, linewidths=0.5)
ax.set_title('Pollutant Correlation Matrix')

# 4d. Delhi PM2.5 time-series – shows seasonal patterns and spikes
ax = axes[1, 1]
delhi = df[df['city'] == 'Delhi'].set_index('date')['pm2_5']
delhi.plot(ax=ax, color='purple', linewidth=0.8)
ax.set_title('Delhi – Daily PM2.5 Over Time')
ax.set_xlabel('Date')
ax.set_ylabel('PM2.5')

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print("EDA plots saved → eda_plots.png")

---
## 5. Feature Engineering

Raw daily readings alone are not sufficient for a good predictive model. We create:

- **Temporal features**: day-of-week, month, quarter, day-of-year (capture seasonality)
- **Lag features** (1, 7, 30 days): air quality today depends heavily on recent readings
- **Rolling statistics** (7-day, 30-day mean and std): smooth short-term noise

> **Target city: Delhi** — largest city with most complete data coverage

In [ ]:
TARGET_CITY = 'Delhi'
TARGET_COL  = 'pm2_5'

city_df = df[df['city'] == TARGET_CITY].copy()
city_df = city_df.set_index('date').sort_index()
city_df = city_df[[TARGET_COL, 'co', 'no', 'no2', 'o3', 'so2', 'pm10', 'nh3', 'aqi']]

# Temporal features to capture weekly/seasonal patterns
city_df['day_of_week']  = city_df.index.dayofweek
city_df['month']        = city_df.index.month
city_df['quarter']      = city_df.index.quarter
city_df['day_of_year']  = city_df.index.dayofyear

# Lag features: past values as predictors for future values
for lag in [1, 7, 30]:
    city_df[f'pm2_5_lag_{lag}'] = city_df[TARGET_COL].shift(lag)

# Rolling statistics: moving average and volatility
city_df['pm2_5_roll7_mean']  = city_df[TARGET_COL].rolling(7).mean()
city_df['pm2_5_roll7_std']   = city_df[TARGET_COL].rolling(7).std()
city_df['pm2_5_roll30_mean'] = city_df[TARGET_COL].rolling(30).mean()

# Drop rows with NaN introduced by lag/rolling operations
city_df = city_df.dropna()
print(f"Feature-engineered shape: {city_df.shape}")
city_df.head()

---
## 6. Train-Test Split & Scaling

For time-series data we use a **chronological split** (no shuffling) to avoid data leakage — future data must never be used to train the model. We use an 80/20 split and apply `MinMaxScaler` to normalise features to [0, 1] range, which helps gradient-based models converge faster.

In [ ]:
feature_cols = [c for c in city_df.columns if c != TARGET_COL]
X = city_df[feature_cols]
y = city_df[TARGET_COL]

# Chronological split – critical for time-series to prevent future data leakage
split_idx  = int(len(city_df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# Fit scaler only on training data; transform both train and test
scaler     = MinMaxScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"Training period : {X_train.index[0].date()} → {X_train.index[-1].date()}")
print(f"Testing period  : {X_test.index[0].date()}  → {X_test.index[-1].date()}")

---
## 7. Model Training & Evaluation

We compare **8 models** covering classical ML, time-series, and deep learning approaches:

| # | Model | Type |
|---|-------|------|
| 1 | Linear Regression | Baseline |
| 2 | Random Forest | Ensemble |
| 3 | Gradient Boosting | Ensemble |
| 4 | **ARIMA** | Time-series |
| 5 | **SARIMA** | Time-series (seasonal) |
| 6 | Holt-Winters | Exponential Smoothing |
| 7 | Prophet | Bayesian Additive |
| 8 | LSTM | Deep Learning |

Evaluation metrics used:
- **MAE** (Mean Absolute Error) — average absolute deviation
- **RMSE** (Root Mean Squared Error) — penalises large errors more
- **R²** (Coefficient of Determination) — explained variance (1.0 = perfect)

In [ ]:
# Utility function to compute and print metrics for any model's predictions
def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f"  {name:<35}  MAE={mae:7.3f}  RMSE={rmse:7.3f}  R²={r2:6.4f}")
    return {'Model': name, 'MAE': round(mae,4), 'RMSE': round(rmse,4), 'R2': round(r2,4)}

results   = []
preds_all = {}

In [ ]:
# ── 7.1 Linear Regression (Baseline) ──
# Simple model to establish a performance baseline.
# If complex models barely beat this, the problem may be linear in nature.
lr = LinearRegression()
lr.fit(X_train_sc, y_train)
lr_pred = lr.predict(X_test_sc)
results.append(evaluate("Linear Regression", y_test, lr_pred))
preds_all['Linear Regression'] = lr_pred

In [ ]:
# ── 7.2 Random Forest ──
# Ensemble of decision trees using bagging. Robust to outliers and handles
# non-linear relationships well. n_jobs=-1 uses all available CPU cores.
rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train_sc, y_train)
rf_pred = rf.predict(X_test_sc)
results.append(evaluate("Random Forest", y_test, rf_pred))
preds_all['Random Forest'] = rf_pred

In [ ]:
# ── 7.3 Gradient Boosting ──
# Sequentially builds trees to correct previous errors (boosting).
# Lower learning_rate with more trees often yields better generalisation.
gb = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=42)
gb.fit(X_train_sc, y_train)
gb_pred = gb.predict(X_test_sc)
results.append(evaluate("Gradient Boosting", y_test, gb_pred))
preds_all['Gradient Boosting'] = gb_pred

In [ ]:
# ── 7.4 ARIMA (AutoRegressive Integrated Moving Average) ──
# ARIMA(p,d,q) models univariate time series:
#   p=5 → use 5 past lags (AR part)
#   d=1 → first-difference to make series stationary
#   q=2 → use 2 past forecast errors (MA part)
# Best for non-seasonal time series with trends.
train_series = y_train.values
print("Fitting ARIMA(5,1,2)...")
arima_model  = ARIMA(train_series, order=(5, 1, 2)).fit()
arima_pred   = arima_model.forecast(steps=len(y_test))
results.append(evaluate("ARIMA(5,1,2)", y_test, arima_pred))
preds_all['ARIMA'] = arima_pred
print("ARIMA fitted successfully!")

In [ ]:
# ── 7.5 SARIMA (Seasonal ARIMA) ──
# Extends ARIMA with seasonal components: SARIMA(p,d,q)(P,D,Q,s)
#   s=7 → weekly seasonality (air quality varies day-to-day in a week)
#   Seasonal order (1,1,1,7) handles weekly cycles
# More powerful than plain ARIMA when data has periodic patterns.
print("Fitting SARIMA(1,1,1)(1,1,1,7)... (this may take a moment)")
sarima_model = SARIMAX(
    train_series,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7)
).fit(disp=False)
sarima_pred  = sarima_model.forecast(steps=len(y_test))
results.append(evaluate("SARIMA(1,1,1)(1,1,1,7)", y_test, sarima_pred))
preds_all['SARIMA'] = sarima_pred
print("SARIMA fitted successfully!")

In [ ]:
# ── 7.6 Holt-Winters Exponential Smoothing ──
# Triple exponential smoothing handles trend + seasonal components.
# Uses additive model here; switch to 'mul' for multiplicative seasonality.
hw_model = ExponentialSmoothing(train_series, trend='add', seasonal='add', seasonal_periods=7).fit()
hw_pred  = hw_model.forecast(len(y_test))
results.append(evaluate("Holt-Winters", y_test, hw_pred))
preds_all['Holt-Winters'] = hw_pred

In [ ]:
# ── 7.7 Prophet ──
# Facebook Prophet decomposes time series into trend + seasonality + holiday effects.
# Requires columns named 'ds' (datestamp) and 'y' (target).
# Particularly effective for data with strong seasonal effects.
if PROPHET_AVAILABLE:
    prophet_train = pd.DataFrame({'ds': y_train.index, 'y': y_train.values})
    prophet_model = Prophet(
        daily_seasonality=True,
        weekly_seasonality=True,
        yearly_seasonality=True
    )
    prophet_model.fit(prophet_train)
    future        = prophet_model.make_future_dataframe(periods=len(y_test))
    forecast      = prophet_model.predict(future)
    prophet_pred  = forecast['yhat'].values[-len(y_test):]
    results.append(evaluate("Prophet", y_test, prophet_pred))
    preds_all['Prophet'] = prophet_pred
else:
    print("Skipping Prophet (not installed).")

In [ ]:
# ── 7.8 LSTM (Long Short-Term Memory) ──
# Deep learning model specifically designed for sequential/temporal data.
# Uses a sliding window of SEQ_LEN=30 past days to predict the next day.
# Dropout layers reduce overfitting; EarlyStopping halts training when val_loss stagnates.
if KERAS_AVAILABLE:
    SEQ_LEN = 30

    def make_sequences(arr, seq_len):
        """Create overlapping windows of length seq_len for LSTM input."""
        X_, y_ = [], []
        for i in range(seq_len, len(arr)):
            X_.append(arr[i-seq_len:i])
            y_.append(arr[i])
        return np.array(X_), np.array(y_)

    pm_scaler    = MinMaxScaler()
    pm_scaled    = pm_scaler.fit_transform(city_df[[TARGET_COL]])
    train_sc_arr = pm_scaled[:split_idx]
    test_sc_arr  = pm_scaled[split_idx - SEQ_LEN:]  # include look-back window

    Xl_train, yl_train = make_sequences(train_sc_arr, SEQ_LEN)
    Xl_test,  yl_test  = make_sequences(test_sc_arr,  SEQ_LEN)

    lstm_model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(SEQ_LEN, 1)),
        Dropout(0.2),
        LSTM(32),
        Dropout(0.2),
        Dense(1)
    ])
    lstm_model.compile(optimizer='adam', loss='mse')

    early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    lstm_model.fit(
        Xl_train, yl_train,
        epochs=50, batch_size=32,
        validation_split=0.1,
        callbacks=[early_stop],
        verbose=0
    )

    lstm_pred_sc = lstm_model.predict(Xl_test, verbose=0)
    lstm_pred    = pm_scaler.inverse_transform(lstm_pred_sc).flatten()
    y_test_lstm  = pm_scaler.inverse_transform(yl_test.reshape(-1,1)).flatten()

    results.append(evaluate("LSTM", y_test_lstm, lstm_pred))
    preds_all['LSTM'] = lstm_pred
else:
    print("Skipping LSTM (TensorFlow not installed).")

---
## 8. Hyperparameter Tuning

Default hyperparameters are rarely optimal. We use:
- **GridSearchCV** for Random Forest (exhaustive search over a small grid)
- **RandomizedSearchCV** for Gradient Boosting (random sampling over a larger space)

Both use cross-validation internally (cv=3 folds) to estimate generalisation performance. The best parameters are then used to retrain on the full training set and evaluate on the held-out test set.

In [ ]:
# ── 8.1 Hyperparameter Tuning: Random Forest (GridSearchCV) ──
# We search over key parameters:
#   n_estimators  → number of trees in the forest
#   max_depth     → max depth per tree (None = unlimited)
#   min_samples_split → minimum samples required to split an internal node
print("Running GridSearchCV for Random Forest (this may take a few minutes)...")

rf_param_grid = {
    'n_estimators'     : [100, 200],
    'max_depth'        : [None, 10, 20],
    'min_samples_split': [2, 5]
}

rf_grid = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    rf_param_grid,
    cv=3,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1
)
rf_grid.fit(X_train_sc, y_train)

print(f"\nBest RF params : {rf_grid.best_params_}")
print(f"Best CV RMSE   : {np.sqrt(-rf_grid.best_score_):.3f}")

# Evaluate the tuned model on the test set
rf_tuned      = rf_grid.best_estimator_
rf_tuned_pred = rf_tuned.predict(X_test_sc)
results.append(evaluate("RF Tuned (GridSearch)", y_test, rf_tuned_pred))
preds_all['RF Tuned'] = rf_tuned_pred

In [ ]:
# ── 8.2 Hyperparameter Tuning: Gradient Boosting (RandomizedSearchCV) ──
# RandomizedSearchCV samples n_iter combinations from the parameter distributions,
# which is much faster than exhaustive grid search for large search spaces.
# Particularly useful when we want to tune learning_rate, depth, and subsampling together.
print("Running RandomizedSearchCV for Gradient Boosting...")

gb_param_dist = {
    'n_estimators'  : [100, 200, 300, 500],
    'learning_rate' : [0.01, 0.05, 0.1, 0.2],
    'max_depth'     : [3, 4, 5, 6],
    'subsample'     : [0.7, 0.8, 1.0],
    'min_samples_split': [2, 5, 10]
}

gb_random = RandomizedSearchCV(
    GradientBoostingRegressor(random_state=42),
    gb_param_dist,
    n_iter=20,
    cv=3,
    scoring='neg_mean_squared_error',
    random_state=42,
    n_jobs=-1,
    verbose=1
)
gb_random.fit(X_train_sc, y_train)

print(f"\nBest GB params : {gb_random.best_params_}")
print(f"Best CV RMSE   : {np.sqrt(-gb_random.best_score_):.3f}")

# Evaluate the tuned model on the test set
gb_tuned      = gb_random.best_estimator_
gb_tuned_pred = gb_tuned.predict(X_test_sc)
results.append(evaluate("GB Tuned (RandomSearch)", y_test, gb_tuned_pred))
preds_all['GB Tuned'] = gb_tuned_pred

In [ ]:
# ── 8.3 ARIMA Order Selection (AIC-based) ──
# We compare several (p,d,q) combinations and select the one with lowest AIC.
# AIC (Akaike Information Criterion) balances model fit vs. complexity;
# lower AIC = better trade-off.
print("Searching for best ARIMA order using AIC...")

best_aic   = np.inf
best_order = (5, 1, 2)  # default fallback
best_arima = None

arima_candidates = [
    (1,1,1), (2,1,1), (2,1,2),
    (3,1,1), (3,1,2), (5,1,2)
]

for order in arima_candidates:
    try:
        m = ARIMA(train_series, order=order).fit()
        if m.aic < best_aic:
            best_aic   = m.aic
            best_order = order
            best_arima = m
        print(f"  ARIMA{order}  AIC={m.aic:.2f}")
    except Exception as e:
        print(f"  ARIMA{order}  FAILED: {e}")

print(f"\n✅ Best ARIMA order: {best_order}  (AIC={best_aic:.2f})")
arima_tuned_pred = best_arima.forecast(steps=len(y_test))
results.append(evaluate(f"ARIMA{best_order} Tuned", y_test, arima_tuned_pred))
preds_all[f'ARIMA Tuned'] = arima_tuned_pred

In [ ]:
# ── 8.4 SARIMA Order Selection ──
# Similarly, we compare seasonal ARIMA configurations and pick the best AIC.
# Seasonal period s=7 (weekly). We vary the non-seasonal and seasonal orders.
print("Searching for best SARIMA order using AIC...")

best_saic    = np.inf
best_sorder  = ((1,1,1),(1,1,1,7))
best_sarima  = None

sarima_candidates = [
    ((1,1,1),(0,1,1,7)),
    ((1,1,1),(1,1,1,7)),
    ((2,1,1),(1,1,1,7)),
    ((1,1,2),(1,1,1,7)),
]

for (order, s_order) in sarima_candidates:
    try:
        m = SARIMAX(train_series, order=order, seasonal_order=s_order).fit(disp=False)
        if m.aic < best_saic:
            best_saic   = m.aic
            best_sorder = (order, s_order)
            best_sarima = m
        print(f"  SARIMA{order}x{s_order}  AIC={m.aic:.2f}")
    except Exception as e:
        print(f"  SARIMA{order}x{s_order}  FAILED: {e}")

print(f"\n✅ Best SARIMA: {best_sorder}  (AIC={best_saic:.2f})")
sarima_tuned_pred = best_sarima.forecast(steps=len(y_test))
results.append(evaluate(f"SARIMA Tuned", y_test, sarima_tuned_pred))
preds_all['SARIMA Tuned'] = sarima_tuned_pred

---
## 9. Results Comparison

All models are ranked by RMSE (lower is better). We also visualize MAE, RMSE, and R² side by side to get a holistic view of performance. R² closest to 1.0 indicates the best-fitting model.

In [ ]:
# Compile all results into a sorted DataFrame
results_df = pd.DataFrame(results).sort_values('RMSE')
print("\n=== Model Comparison (sorted by RMSE) ===")
print(results_df.to_string(index=False))

best_model_name = results_df.iloc[0]['Model']
print(f"\n🏆 Best model: {best_model_name}")

In [ ]:
# Visual comparison of all models across the three metrics
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
metrics = ['MAE', 'RMSE', 'R2']
colors  = ['#4C72B0', '#DD8452', '#55A868']

for ax, metric, color in zip(axes, metrics, colors):
    results_df.sort_values(metric, ascending=(metric != 'R2')).plot(
        kind='barh', x='Model', y=metric, ax=ax, color=color, legend=False
    )
    ax.set_title(f'{metric} Comparison')
    ax.set_xlabel(metric)

plt.suptitle('Model Performance Comparison (All Models including Tuned)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Visualise Actual vs Predicted

Plotting actual vs. predicted values is the most intuitive way to assess model quality. A good model should:
- Closely track actual values
- Correctly capture peaks and troughs (pollution spikes)
- Not exhibit large systematic biases

In [ ]:
# Plot actual vs predicted for each model – stored for visual inspection
test_dates = y_test.index

fig, axes = plt.subplots(len(preds_all), 1, figsize=(14, 4 * len(preds_all)), sharex=False)
if len(preds_all) == 1:
    axes = [axes]

for ax, (name, pred) in zip(axes, preds_all.items()):
    n = min(len(y_test), len(pred))
    ax.plot(test_dates[:n], y_test.values[:n], label='Actual', color='black', linewidth=1.5)
    ax.plot(test_dates[:n], pred[:n],           label=name,   color='royalblue', linewidth=1, alpha=0.85)
    ax.set_title(f'{name} – Actual vs Predicted (PM2.5, Delhi)')
    ax.set_ylabel('PM2.5 µg/m³')
    ax.legend()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved → actual_vs_predicted.png")

---
## 11. Feature Importance (Random Forest & Gradient Boosting)

Tree-based models provide feature importance scores that reveal **which input variables most influence PM2.5 predictions**. High-importance features should be retained; low-importance ones can potentially be dropped to simplify the model.

We compare importances from both the default and the tuned models.

In [ ]:
# Feature importance from default and tuned tree models
fig, axes = plt.subplots(2, 2, figsize=(18, 10))

model_pairs = [
    (rf,       'Random Forest (Default)',    axes[0, 0]),
    (gb,       'Gradient Boosting (Default)',axes[0, 1]),
    (rf_tuned, 'Random Forest (Tuned)',      axes[1, 0]),
    (gb_tuned, 'GB (Tuned)',                 axes[1, 1]),
]

for model, title, ax in model_pairs:
    imp = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=True)
    imp.tail(15).plot(kind='barh', ax=ax, color='teal')
    ax.set_title(f'{title} – Feature Importance')
    ax.set_xlabel('Importance Score')

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 12. 7-Day Future Forecast

We use Prophet (if available) to forecast PM2.5 for the next 7 days beyond the test period. The shaded band represents the 95% confidence interval, which widens as we look further into the future — reflecting increased uncertainty.

In [ ]:
# 7-day future forecast with confidence intervals using Prophet
FORECAST_DAYS = 7

if PROPHET_AVAILABLE:
    future_ext   = prophet_model.make_future_dataframe(periods=len(y_test) + FORECAST_DAYS)
    forecast_ext = prophet_model.predict(future_ext)
    future_only  = forecast_ext.tail(FORECAST_DAYS)[['ds', 'yhat', 'yhat_lower', 'yhat_upper']]
    print("=== Prophet 7-Day Forecast (PM2.5, Delhi) ===")
    print(future_only.to_string(index=False))

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.fill_between(future_only['ds'], future_only['yhat_lower'], future_only['yhat_upper'],
                    alpha=0.3, color='steelblue', label='95% CI')
    ax.plot(future_only['ds'], future_only['yhat'], marker='o', color='navy', label='Forecast')
    ax.set_title('7-Day PM2.5 Forecast – Delhi (Prophet)')
    ax.set_ylabel('PM2.5 µg/m³')
    ax.legend()
    plt.tight_layout()
    plt.savefig('7day_forecast.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Prophet not available – skipping 7-day forecast visualisation.")

---
## 13. Predict Using Dummy (Synthetic) Data

To demonstrate real-world inference, we create synthetic input data that mimics sensor readings a monitoring station might send. This shows how the trained pipeline handles new, unseen inputs — a critical requirement for deploying the model in production via the Flask API.

The dummy records represent:
- A **moderately polluted day** (representative of Delhi in winter)
- A **low pollution day** (clear weather conditions)
- A **high pollution spike** (smog/festival conditions)

In [ ]:
# ── Create three representative synthetic input records ──
# Each dict simulates a single day's sensor readings for Delhi.
# Values are based on realistic ranges observed in the dataset.

dummy_inputs = [
    {
        'label'             : 'Moderate Pollution Day',
        'co'                : 900.0,
        'no'                : 8.5,
        'no2'               : 32.0,
        'o3'                : 25.0,
        'so2'               : 18.0,
        'pm10'              : 120.0,
        'nh3'               : 12.0,
        'aqi'               : 3,
        'day_of_week'       : 2,
        'month'             : 11,
        'quarter'           : 4,
        'day_of_year'       : 320,
        'pm2_5_lag_1'       : 95.0,
        'pm2_5_lag_7'       : 88.0,
        'pm2_5_lag_30'      : 75.0,
        'pm2_5_roll7_mean'  : 91.0,
        'pm2_5_roll7_std'   : 10.2,
        'pm2_5_roll30_mean' : 82.0,
    },
    {
        'label'             : 'Low Pollution Day (Clear Weather)',
        'co'                : 420.0,
        'no'                : 2.0,
        'no2'               : 14.0,
        'o3'                : 40.0,
        'so2'               : 6.0,
        'pm10'              : 35.0,
        'nh3'               : 4.0,
        'aqi'               : 1,
        'day_of_week'       : 6,
        'month'             : 7,
        'quarter'           : 3,
        'day_of_year'       : 195,
        'pm2_5_lag_1'       : 28.0,
        'pm2_5_lag_7'       : 31.0,
        'pm2_5_lag_30'      : 42.0,
        'pm2_5_roll7_mean'  : 30.0,
        'pm2_5_roll7_std'   : 4.5,
        'pm2_5_roll30_mean' : 38.0,
    },
    {
        'label'             : 'High Pollution Spike (Smog/Festival)',
        'co'                : 2800.0,
        'no'                : 25.0,
        'no2'               : 80.0,
        'o3'                : 10.0,
        'so2'               : 55.0,
        'pm10'              : 380.0,
        'nh3'               : 38.0,
        'aqi'               : 5,
        'day_of_week'       : 4,
        'month'             : 10,
        'quarter'           : 4,
        'day_of_year'       : 300,
        'pm2_5_lag_1'       : 310.0,
        'pm2_5_lag_7'       : 180.0,
        'pm2_5_lag_30'      : 120.0,
        'pm2_5_roll7_mean'  : 250.0,
        'pm2_5_roll7_std'   : 60.0,
        'pm2_5_roll30_mean' : 160.0,
    }
]

# Convert to DataFrame aligned with training feature columns
dummy_df     = pd.DataFrame(dummy_inputs)
dummy_labels = dummy_df['label'].tolist()
dummy_X      = dummy_df[feature_cols]  # keep only model features
dummy_X_sc   = scaler.transform(dummy_X)

print("Dummy input records:")
print(dummy_X.to_string())

In [ ]:
# ── Run predictions through all ML models on the dummy data ──
# Each model produces a PM2.5 forecast for the three synthetic scenarios.

dummy_results = {}

ml_models = {
    'Linear Regression' : lr,
    'Random Forest'     : rf,
    'Gradient Boosting' : gb,
    'RF Tuned'          : rf_tuned,
    'GB Tuned'          : gb_tuned,
}

for model_name, model in ml_models.items():
    preds = model.predict(dummy_X_sc)
    dummy_results[model_name] = preds

# Build a display table
dummy_result_df = pd.DataFrame(dummy_results, index=dummy_labels).round(2)
dummy_result_df.index.name = 'Scenario'
print("\n=== Dummy Data Predictions (PM2.5 µg/m³) ===")
print(dummy_result_df.to_string())

In [ ]:
# Visualise dummy predictions – grouped bar chart comparing model outputs
fig, ax = plt.subplots(figsize=(12, 6))

x        = np.arange(len(dummy_labels))
n_models = len(dummy_results)
width    = 0.15
colors   = plt.cm.tab10(np.linspace(0, 1, n_models))

for i, (model_name, preds) in enumerate(dummy_results.items()):
    ax.bar(x + i * width, preds, width, label=model_name, color=colors[i], edgecolor='white')

ax.set_xticks(x + width * (n_models - 1) / 2)
ax.set_xticklabels(dummy_labels, rotation=10, ha='right')
ax.set_ylabel('Predicted PM2.5 (µg/m³)')
ax.set_title('Predictions on Synthetic Dummy Data – Model Comparison')
ax.legend(loc='upper right')

# Reference lines for WHO guidelines (15 µg/m³ annual mean) and Indian NAAQS (60 µg/m³)
ax.axhline(15,  color='green',  linestyle='--', linewidth=1, alpha=0.6, label='WHO guideline (15)')
ax.axhline(60,  color='orange', linestyle='--', linewidth=1, alpha=0.6, label='NAAQS standard (60)')
ax.legend()

plt.tight_layout()
plt.savefig('dummy_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Dummy predictions chart saved → dummy_predictions.png")

In [ ]:
# Interpret predictions using AQI category thresholds for PM2.5 (Indian CPCB standard)
def pm25_to_aqi_category(pm25_val):
    """Convert predicted PM2.5 value to an AQI category label."""
    if pm25_val <= 30:
        return 'Good / Satisfactory'
    elif pm25_val <= 60:
        return 'Moderate'
    elif pm25_val <= 90:
        return 'Poor'
    elif pm25_val <= 120:
        return 'Very Poor'
    else:
        return 'Severe'

print("\n=== Predicted PM2.5 + AQI Category Interpretation ===")
best_ml_model = 'GB Tuned'  # use tuned Gradient Boosting for interpretation

for scenario, pred_val in zip(dummy_labels, dummy_results[best_ml_model]):
    category = pm25_to_aqi_category(pred_val)
    print(f"  {scenario:<40}  PM2.5={pred_val:6.2f}  → {category}")

---
## 14. Save Best Model & Artifacts

We persist the trained models, feature scaler, and feature column list to disk. This allows the **Flask API** to load models at startup without retraining. We save:
- `random_forest.pkl` and `gradient_boosting.pkl` — default models
- `rf_tuned.pkl` and `gb_tuned.pkl` — hyperparameter-tuned models
- `feature_scaler.pkl` — fitted MinMaxScaler
- `feature_cols.json` — ordered feature list for consistent input construction
- `model_results.csv` — benchmark table

In [ ]:
os.makedirs('model_artifacts', exist_ok=True)

# Save all trained models
joblib.dump(rf,       'model_artifacts/random_forest.pkl')
joblib.dump(gb,       'model_artifacts/gradient_boosting.pkl')
joblib.dump(rf_tuned, 'model_artifacts/rf_tuned.pkl')
joblib.dump(gb_tuned, 'model_artifacts/gb_tuned.pkl')
joblib.dump(scaler,   'model_artifacts/feature_scaler.pkl')

# Save feature column list (ensures API uses identical feature ordering)
with open('model_artifacts/feature_cols.json', 'w') as f:
    json.dump(feature_cols, f)

# Save full results table
results_df.to_csv('model_artifacts/model_results.csv', index=False)

if PROPHET_AVAILABLE:
    joblib.dump(prophet_model, 'model_artifacts/prophet_model.pkl')

print("✅ Artifacts saved to model_artifacts/")
for f in sorted(os.listdir('model_artifacts')):
    print(f"   {f}")

---
## 15. Summary & Model Comparison

In [ ]:
print("=" * 65)
print("  AIR QUALITY PREDICTION – PROJECT SUMMARY")
print("=" * 65)
print(f"  Dataset     : {df_raw.shape[0]:,} records | {df_raw['city'].nunique()} cities")
print(f"  Target      : PM2.5 (Delhi)")
print(f"  Train / Test: {split_idx} / {len(y_test)} samples")
print()
print("  Models compared:")
for _, row in results_df.iterrows():
    star = '  ★ BEST' if row['Model'] == best_model_name else ''
    print(f"    {row['Model']:<38} R²={row['R2']:.4f}{star}")
print()
print(f"  🏆 Best Model : {best_model_name}")
print(f"     MAE  = {results_df.iloc[0]['MAE']}")
print(f"     RMSE = {results_df.iloc[0]['RMSE']}")
print(f"     R²   = {results_df.iloc[0]['R2']}")
print("=" * 65)

---
## 16. Limitations

While this project demonstrates a comprehensive multi-model approach to air quality prediction, several important limitations should be acknowledged:

### 16.1 Data Limitations
- **Single city focus**: Models are trained and evaluated specifically on Delhi data. Performance may degrade significantly when applied to other cities with different pollution profiles, meteorological conditions, or urban layouts.
- **Missing meteorological data**: Key weather variables such as wind speed, wind direction, temperature, humidity, and atmospheric pressure are absent. These strongly influence pollutant dispersion and are standard inputs in operational air quality models.
- **Sentinel value imputation**: Replacing -200 (missing sensor readings) with column means is a simple strategy. More sophisticated imputation (KNN imputation, interpolation from neighbouring stations) may produce more realistic estimates.
- **Temporal gaps**: If the dataset has irregular time steps or multiple consecutive missing days, lag features and rolling statistics will contain compounded imputation errors.

### 16.2 Model Limitations
- **ARIMA/SARIMA are univariate**: They only use past PM2.5 values and cannot incorporate other pollutant readings or weather data as exogenous inputs without switching to ARIMAX/SARIMAX.
- **ARIMA non-stationarity assumption**: The differencing parameter d=1 may not always be sufficient; the series should ideally pass an ADF stationarity test before selecting ARIMA orders.
- **LSTM training instability**: Deep learning models are sensitive to hyperparameters (learning rate, hidden units, dropout rate). Results may vary across runs without a fixed random seed throughout.
- **Short SARIMA seasonal period**: Using s=7 (weekly) may not capture the dominant annual seasonal cycle in PM2.5, which peaks in October–February (winter smog season).
- **Hyperparameter search scope**: GridSearchCV/RandomizedSearchCV cover a limited parameter space. More advanced methods like Bayesian optimisation (Optuna, Hyperopt) could find better configurations.

### 16.3 Deployment Limitations
- **No real-time data ingestion**: The Flask API predicts from manually entered feature values; integrating live sensor feeds would require a streaming pipeline (e.g., Kafka, MQTT).
- **Model drift**: Air quality patterns change over time due to policy changes, new emission sources, or climate change. Models should be retrained periodically.
- **No uncertainty quantification**: Most ML models (except Prophet) return point estimates. Prediction intervals (e.g., quantile regression, conformal prediction) would make the system more useful for decision-making.
- **Scalability**: The current codebase is single-city and single-target. Scaling to 100+ cities with multiple pollutants requires architectural changes.

---
## 17. Conclusion

This project successfully built an end-to-end air quality prediction pipeline for PM2.5 forecasting in Delhi, India, using a diverse set of machine learning and time-series models.

### Key Findings

1. **Ensemble tree-based models outperformed linear and time-series models** in this dataset. Random Forest and Gradient Boosting (both default and tuned) achieved the highest R² scores, benefiting from the rich feature set of lag variables and rolling statistics.

2. **Hyperparameter tuning provided measurable improvements** over default settings, particularly for Gradient Boosting where learning rate and subsample fraction significantly affected performance. ARIMA order selection via AIC also led to a better-calibrated time-series model.

3. **ARIMA and SARIMA demonstrated moderate performance** as univariate models, capturing temporal autocorrelation but unable to leverage the multi-variate pollutant information available to tree-based models. SARIMA's weekly seasonality component improved upon plain ARIMA.

4. **Prophet provided reasonable forecasts with uncertainty bounds**, making it well-suited for communicating risk ranges to non-technical stakeholders and for 7-day ahead forecasting.

5. **Feature engineering was critical**: Lag features (especially 1-day and 7-day lags) and rolling statistics were consistently among the top predictors, confirming the strong temporal autocorrelation in pollution data.

6. **Dummy data inference confirmed deployment readiness**: The trained models correctly ranked the three synthetic scenarios (low, moderate, high pollution) in the expected order, validating the inference pipeline used in the Flask API.

### Recommendations for Future Work

- Incorporate **meteorological covariates** (wind, temperature, humidity) to improve forecast accuracy
- Explore **XGBoost / LightGBM** as faster and often more accurate alternatives to sklearn's Gradient Boosting
- Implement **online/incremental learning** to continuously update models as new sensor data arrives
- Deploy to a cloud platform (GCP, AWS, Azure) with automated retraining pipelines
- Add **explainability tools** (SHAP values) to help policymakers understand what drives pollution spikes
- Extend the multi-city analysis to build **city-agnostic transfer learning** models

### Impact

Accurate air quality forecasting has direct public health implications. By providing advance warnings of high-pollution days, models like these can inform:
- Citizens to reduce outdoor activities on unhealthy air days
- Municipal authorities to implement emergency traffic/industrial controls
- Hospitals to prepare for increased respiratory emergency admissions

This project forms a strong foundation for real-world environmental AI applications.

---
*Built with ❤️ | INLIGHN TECH | Air Quality Prediction Project*